# Download Evaluation Datasets

This notebook downloads and formats the evaluation datasets used in the think-overflow experiments.

**Code benchmarks:**
- EvalPlus (HumanEval+ and MBPP+)
- LiveCodeBench
- BigCodeBench
- CRUXEval (code understanding)
- Code Contests (competitive programming)

**Other benchmarks:**
- GSM8K
- MATH-500
- GPQA

In [5]:
import json
import re
import random
import statistics
from pathlib import Path

from datasets import Dataset, concatenate_datasets, load_dataset
from huggingface_hub import hf_hub_download
from llm_cgr import save_jsonl

# this notebook lives in data/ so output dirs are relative to here
CODE_DIR = Path("./code")
CRUX_DIR = Path("./crux")
MATH_DIR = Path("./math")
REASONING_DIR = Path("./reasoning")

# create all directories
for d in [CODE_DIR, CRUX_DIR, MATH_DIR, REASONING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [6]:
# commit hashes for reproducibility
REVISIONS = {
    "humanevalplus": "d32357c",
    "mbppplus": "b2d74c9",
    "livecodebench": "0fe84c3",
    "bigcodebench": "b74c0d0",
    "gsm8k": "cc7b047",
    "math500": "6e4ed1a",
    "gpqa": "fa6a028",
    "cruxeval": "b96af04",
    # pin after first run to lock dataset version
    "code_contests": "802411c",
}

In [7]:
# maximum prompt length to keep — prompts over this are removed at download time.
# our models have a 32768-token context window; we need room for thinking tokens
# (up to 28664 in experiments) plus an answer, so we cap prompts at half the window.
# this uses a chars-based approximation (1 token ≈ 4 chars for code/text).
MAX_PROMPT_TOKENS = 4_096
MAX_PROMPT_CHARS = MAX_PROMPT_TOKENS * 4  # approximate character threshold


def _estimate_tokens(text: str) -> int:
    """Estimate token count from character length (1 token ≈ 4 chars).

    Returns an integer estimate — accurate enough to catch grossly overlong prompts.
    """
    return len(text) // 4


def _filter_by_length(
    records: list[dict],
    dataset_name: str,
) -> list[dict]:
    """Remove records whose prompt exceeds MAX_PROMPT_TOKENS (estimated).

    Prints a summary of how many records were dropped and why.

    Returns the filtered list.
    """
    before = len(records)
    filtered = [
        r for r in records if _estimate_tokens(r["prompt"]) <= MAX_PROMPT_TOKENS
    ]
    dropped = before - len(filtered)
    if dropped:
        print(f"  filtered {dropped}/{before} overlong prompts from {dataset_name}")
    else:
        print(f"  {dataset_name}: all {before} prompts within token limit")
    return filtered

## Code Benchmarks

### EvalPlus

HumanEval+ and MBPP+ with augmented test suites — 164 + 378 = 542 problems total, evaluated by executing generated functions against test cases.

Sources: [evalplus/humanevalplus](https://huggingface.co/datasets/evalplus/humanevalplus) · [evalplus/mbppplus](https://huggingface.co/datasets/evalplus/mbppplus)

In [8]:
# load humaneval+ dataset
humaneval = load_dataset(
    "evalplus/humanevalplus",
    split="test",
    revision=REVISIONS["humanevalplus"],
)
print(f"loaded {len(humaneval)} humaneval+ problems")
print(f"columns: {humaneval.column_names}")

loaded 164 humaneval+ problems
columns: ['task_id', 'prompt', 'canonical_solution', 'entry_point', 'test']


In [9]:
# load mbpp+ dataset
mbpp = load_dataset(
    "evalplus/mbppplus",
    split="test",
    revision=REVISIONS["mbppplus"],
)
print(f"loaded {len(mbpp)} mbpp+ problems")
print(f"columns: {mbpp.column_names}")

loaded 378 mbpp+ problems
columns: ['task_id', 'code', 'prompt', 'source_file', 'test_imports', 'test_list', 'test']


In [10]:
# format for our pipeline
# humaneval+ provides: prompt (function signature + docstring), entry_point, test
# mbpp+ provides: code, prompt (description), test (no entry_point, extract from code)

# instructions for instruction-tuned models
HUMANEVAL_INSTRUCTION = "Complete the following Python function:\n\n"
MBPP_FUNCTION_NAME = "\n\nYour function should be named `{entry_point}`."

evalplus_records = []

# add humaneval+ problems
for row in humaneval:
    record = {
        "task_id": row["task_id"],
        "prompt": HUMANEVAL_INSTRUCTION + row["prompt"],
        "entry_point": row["entry_point"],
        "test_code": row["test"],
        "source": "humanevalplus",
    }
    evalplus_records.append(record)

# add mbpp+ problems (extract entry_point from code field)
for row in mbpp:
    # extract function name from the code solution
    code = row["code"]
    match = re.search(r"def\s+(\w+)\s*\(", code)
    entry_point = match.group(1) if match else None

    # append function name requirement to the original prompt
    prompt = row["prompt"] + MBPP_FUNCTION_NAME.format(entry_point=entry_point)

    record = {
        # task_id is an integer in mbpp+; convert to string for consistency
        "task_id": "MBPP/" + str(row["task_id"]),
        "prompt": prompt,
        "entry_point": entry_point,
        "test_code": row["test"],
        "source": "mbppplus",
    }
    evalplus_records.append(record)

evalplus_records = _filter_by_length(evalplus_records, "code/evalplus")
save_jsonl(evalplus_records, str(CODE_DIR / "evalplus.jsonl"))
print(f"saved {len(evalplus_records)} records to code/evalplus.jsonl")

  code/evalplus: all 542 prompts within token limit
saved 542 records to code/evalplus.jsonl


### LiveCodeBench

342 competitive programming problems from recent contests (test5 + test6 releases), continuously updated. Programs read from stdin and write to stdout.

Source: [livecodebench/code_generation_lite](https://huggingface.co/datasets/livecodebench/code_generation_lite)

In [11]:
# load livecodebench dataset (test5 + test6 releases)
# note: datasets 4.0+ removed support for dataset scripts, so we download jsonl directly
# see: https://github.com/LiveCodeBench/LiveCodeBench/issues/107
lcb_datasets = []
for filename in ["test5.jsonl", "test6.jsonl"]:
    jsonl_path = hf_hub_download(
        repo_id="livecodebench/code_generation_lite",
        filename=filename,
        repo_type="dataset",
        revision=REVISIONS["livecodebench"],
    )
    lcb_datasets.append(Dataset.from_json(jsonl_path))

lcb = concatenate_datasets(lcb_datasets)
print(f"loaded {len(lcb)} problems (test5 + test6)")
print(f"columns: {lcb.column_names}")

loaded 342 problems (test5 + test6)
columns: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata']


In [12]:
# format for our pipeline
# livecodebench provides: question_content, public_test_cases (json string)
# note: problems are language-agnostic, so we add a Python instruction
PYTHON_INSTRUCTION = (
    "\n\nWrite a Python program that reads from stdin and writes to stdout."
)

lcb_records = []
for row in lcb:
    # parse test cases from json string
    public_tests = (
        json.loads(row["public_test_cases"]) if row["public_test_cases"] else []
    )

    # extract inputs and expected outputs
    inputs = [t["input"] for t in public_tests]
    expected_outputs = [t["output"] for t in public_tests]

    record = {
        "task_id": row["question_id"],
        "prompt": row["question_content"] + PYTHON_INSTRUCTION,
        "inputs": inputs,
        "expected_outputs": expected_outputs,
    }
    lcb_records.append(record)

lcb_records = _filter_by_length(lcb_records, "code/livecodebench")
save_jsonl(lcb_records, str(CODE_DIR / "livecodebench.jsonl"))
print(f"saved {len(lcb_records)} records to code/livecodebench.jsonl")

  code/livecodebench: all 342 prompts within token limit
saved 342 records to code/livecodebench.jsonl


### BigCodeBench

1,140 function-level tasks testing real-world library usage (pandas, numpy, requests, etc.), evaluated using `unittest.TestCase`. Scored via the official Docker harness.

Source: [bigcode/bigcodebench](https://huggingface.co/datasets/bigcode/bigcodebench)

In [13]:
# load bigcodebench from huggingface — avoids the bigcodebench package which has
# broken transitive dependencies (requires wget, which is not installed)
bcb = load_dataset(
    "bigcode/bigcodebench",
    split="v0.1.2",
    revision=REVISIONS["bigcodebench"],
)
print(f"loaded {len(bcb)} bigcodebench problems")
print(f"columns: {bcb.column_names}")

# format for our pipeline
# bigcodebench provides: task_id, instruct_prompt (natural language instruction),
# entry_point (always "task_func"), test (unittest.TestCase class)
bcb_records = [
    {
        "task_id": row["task_id"],
        "prompt": row["instruct_prompt"],
        "entry_point": row["entry_point"],  # always "task_func"
        "test_code": row["test"],  # unittest.TestCase class
    }
    for row in bcb
]

bcb_records = _filter_by_length(bcb_records, "code/bigcodebench")
save_jsonl(bcb_records, str(CODE_DIR / "bigcodebench.jsonl"))
print(f"saved {len(bcb_records)} records to code/bigcodebench.jsonl")

loaded 1140 bigcodebench problems
columns: ['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs']
  code/bigcodebench: all 1140 prompts within token limit
saved 1140 records to code/bigcodebench.jsonl


### CRUXEval

800 Python functions with two tasks: predict the output given an input (CRUXEval-O), or find an input that produces a given output (CRUXEval-I). Saved as two separate files.

Source: [cruxeval-org/cruxeval](https://huggingface.co/datasets/cruxeval-org/cruxeval)

In [14]:
# load cruxeval dataset (single test split, 800 problems)
cruxeval = load_dataset(
    "cruxeval-org/cruxeval",
    split="test",
    revision=REVISIONS["cruxeval"],
)
print(f"loaded {len(cruxeval)} problems")

loaded 800 problems


In [15]:
# format for our pipeline
# cruxeval provides: code (function), input, output, id
# we create two separate datasets for the two tasks:
#   cruxeval_o: predict output given code + input
#   cruxeval_i: predict input given code + expected output

# --- cruxeval-o: output prediction ---
# prompt matches the original paper's direct (non-cot) format:
#   https://github.com/facebookresearch/cruxeval/blob/main/prompts.py
CRUXEVAL_O_PROMPT = (
    """You are given a Python function and an assertion containing an input to the function. """
    """Complete the assertion with a literal (no unsimplified expressions, no function calls) """
    """containing the output when executing the provided code on the given input, even if the """
    """function is incorrect or incomplete. Do NOT output any extra information. Provide the """
    """full assertion with the correct output in [ANSWER] and [/ANSWER] tags.
"""
    """
Here are 2 examples, showing the expected format.
"""
    """
[PYTHON]
def f(n):
    return n
assert f(17) == ??
[/PYTHON]
"""
    """[ANSWER]
assert f(17) == 17
[/ANSWER]
"""
    """
[PYTHON]
def f(s):
    return s + "a"
assert f("x9j") == ??
[/PYTHON]
"""
    """[ANSWER]
assert f("x9j") == "x9ja"
[/ANSWER]
"""
    """
Now solve the following problem:
"""
    """
[PYTHON]
{code}
assert f({input}) == ??
[/PYTHON]
"""
)

cruxeval_o_records = []
for row in cruxeval:
    # fill in the code and input for this problem
    prompt = CRUXEVAL_O_PROMPT.format(code=row["code"], input=row["input"])
    cruxeval_o_records.append(
        {
            "task_id": "CRUXO/" + str(row["id"]),
            "prompt": prompt,
            "answer": row["output"],  # ground truth output for evaluation
            "task": "output",
        }
    )

cruxeval_o_records = _filter_by_length(cruxeval_o_records, "crux/cruxeval_o")
save_jsonl(cruxeval_o_records, str(CRUX_DIR / "cruxeval_o.jsonl"))
print(f"saved {len(cruxeval_o_records)} records to crux/cruxeval_o.jsonl")

# --- cruxeval-i: input prediction ---
CRUXEVAL_I_PROMPT = (
    """You will be given a function f and an output in the form f(??) == output. Find any """
    """input such that executing f on the input leads to the given output. There may be """
    """multiple answers, but you should only output one. In [ANSWER] and [/ANSWER] tags, """
    """complete the assertion with one such input that will produce the output when executing """
    """the function.
"""
    """
Here are 2 examples, showing the expected format.
"""
    """
[PYTHON]
def f(my_list):
    count = 0
    for i in my_list:
        if len(i) % 2 == 0:
            count += 1
    return count
assert f(??) == 3
[/PYTHON]
"""
    """[ANSWER]
assert f(["mq", "px", "zy"]) == 3
[/ANSWER]
"""
    """
[PYTHON]
def f(s1, s2):
    return s1 + s2
assert f(??) == "banana"
[/PYTHON]
"""
    """[ANSWER]
assert f("ba", "nana") == "banana"
[/ANSWER]
"""
    """
Now solve the following problem:
"""
    """
[PYTHON]
{code}
assert f(??) == {output}
[/PYTHON]
"""
)

cruxeval_i_records = []
for row in cruxeval:
    # fill in the code and expected output for this problem
    prompt = CRUXEVAL_I_PROMPT.format(code=row["code"], output=row["output"])
    cruxeval_i_records.append(
        {
            "task_id": "CRUXI/" + str(row["id"]),
            "prompt": prompt,
            "code": row["code"],  # needed to execute f(predicted) for evaluation
            "answer": row[
                "output"
            ],  # expected output used to verify the predicted input
            "task": "input",
        }
    )

cruxeval_i_records = _filter_by_length(cruxeval_i_records, "crux/cruxeval_i")
save_jsonl(cruxeval_i_records, str(CRUX_DIR / "cruxeval_i.jsonl"))
print(f"saved {len(cruxeval_i_records)} records to crux/cruxeval_i.jsonl")

  crux/cruxeval_o: all 800 prompts within token limit
saved 800 records to crux/cruxeval_o.jsonl
  crux/cruxeval_i: all 800 prompts within token limit
saved 800 records to crux/cruxeval_i.jsonl


### Code Contests

~10,000 competitive programming problems compiled by DeepMind, sourced from Codeforces and similar platforms. Each problem has public test cases for evaluation. Programs read from stdin and write to stdout.

Source: [deepmind/code_contests](https://huggingface.co/datasets/deepmind/code_contests)

In [16]:
# load code_contests test split (~10k competitive programming problems)
code_contests = load_dataset(
    "deepmind/code_contests",
    split="test",
    revision=REVISIONS["code_contests"],
)
print(f"loaded {len(code_contests)} problems")
print(f"columns: {code_contests.column_names}")

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

loaded 165 problems
columns: ['name', 'description', 'public_tests', 'private_tests', 'generated_tests', 'source', 'difficulty', 'solutions', 'incorrect_solutions', 'cf_contest_id', 'cf_index', 'cf_points', 'cf_rating', 'cf_tags', 'is_description_translated', 'untranslated_description', 'time_limit', 'memory_limit_bytes', 'input_file', 'output_file']


In [17]:
# format for our pipeline
# code_contests provides: description (problem statement), public_tests (input/output lists)
# problems are language-agnostic; add python instruction (same pattern as livecodebench)
CODE_CONTESTS_INSTRUCTION = (
    "\n\nWrite a Python program that reads from stdin and writes to stdout."
)

code_contests_records = []
for idx, row in enumerate(code_contests):
    # use public test cases for evaluation (private tests are not available at inference time)
    inputs = row["public_tests"]["input"]
    expected_outputs = row["public_tests"]["output"]

    record = {
        # use index as stable id (names can contain spaces or special characters)
        "task_id": f"CodeContests/{idx}",
        "prompt": row["description"] + CODE_CONTESTS_INSTRUCTION,
        "inputs": inputs,
        "expected_outputs": expected_outputs,
    }
    code_contests_records.append(record)

code_contests_records = _filter_by_length(code_contests_records, "code/code_contests")
save_jsonl(code_contests_records, str(CODE_DIR / "code_contests.jsonl"))
print(f"saved {len(code_contests_records)} records to code/code_contests.jsonl")

  code/code_contests: all 165 prompts within token limit
saved 165 records to code/code_contests.jsonl


## Math Benchmarks

### GSM8K

1,319 grade school math word problems requiring multi-step arithmetic reasoning.

Source: [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k)

In [18]:
# load gsm8k dataset (test split)
gsm8k = load_dataset(
    "openai/gsm8k",
    "main",
    split="test",
    revision=REVISIONS["gsm8k"],
)
print(f"loaded {len(gsm8k)} problems")

# format for our pipeline
# gsm8k provides: question, answer (with #### final_answer format)
gsm8k_records = []
for idx, row in enumerate(gsm8k):
    # extract the final numeric answer after ####
    answer_text = row["answer"]
    match = re.search(r"####\s*(.+)$", answer_text)
    final_answer = match.group(1).strip() if match else answer_text

    record = {
        # gsm8k has no native id; use row index as a stable string id
        "task_id": f"GSM8K/{idx}",
        "prompt": row["question"],
        "answer": final_answer,
    }
    gsm8k_records.append(record)

gsm8k_records = _filter_by_length(gsm8k_records, "math/gsm8k")
save_jsonl(gsm8k_records, str(MATH_DIR / "gsm8k.jsonl"))
print(f"saved {len(gsm8k_records)} records to math/gsm8k.jsonl")

loaded 1319 problems
  math/gsm8k: all 1319 prompts within token limit
saved 1319 records to math/gsm8k.jsonl


### MATH-500

500 competition-level math problems spanning 7 subjects (algebra, geometry, number theory, etc.), sampled from the MATH benchmark.

Source: [HuggingFaceH4/MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)

In [19]:
# load math-500 dataset (test split, 500 competition-level problems)
math500 = load_dataset(
    "HuggingFaceH4/MATH-500",
    split="test",
    revision=REVISIONS["math500"],
)
print(f"loaded {len(math500)} problems")

# format for our pipeline
# math-500 provides: problem, answer (latex expression, no #### prefix)
math500_records = []
for idx, row in enumerate(math500):
    record = {
        # math-500 has no native id; use row index as a stable string id
        "task_id": f"MATH500/{idx}",
        "prompt": row["problem"],
        "answer": row["answer"],
    }
    math500_records.append(record)

math500_records = _filter_by_length(math500_records, "math/math500")
save_jsonl(math500_records, str(MATH_DIR / "math500.jsonl"))
print(f"saved {len(math500_records)} records to math/math500.jsonl")

loaded 500 problems
  math/math500: all 500 prompts within token limit
saved 500 records to math/math500.jsonl


## Reasoning Benchmarks

### GPQA

448 graduate-level multiple-choice questions in biology, physics, and chemistry, written and validated by domain experts. Answer choices are shuffled per question.

Source: [Idavidrein/gpqa](https://huggingface.co/datasets/Idavidrein/gpqa)

In [20]:
# load gpqa dataset (main subset)
# note: gpqa only has a "train" split (the dataset is designed as evaluation data)
gpqa = load_dataset(
    "Idavidrein/gpqa",
    "gpqa_main",
    split="train",
    revision=REVISIONS["gpqa"],
)
print(f"loaded {len(gpqa)} problems")

# format for our pipeline
# gpqa provides: Question, Correct Answer, Incorrect Answer 1/2/3
gpqa_records = []
for idx, row in enumerate(gpqa):
    # collect all choices and shuffle them
    correct = row["Correct Answer"]
    incorrect = [
        row["Incorrect Answer 1"],
        row["Incorrect Answer 2"],
        row["Incorrect Answer 3"],
    ]

    # create shuffled choices with tracked correct index
    choices = [correct] + incorrect
    indices = list(range(4))
    random.seed(idx)  # deterministic per question (stable across python versions)
    random.shuffle(indices)
    shuffled_choices = [choices[i] for i in indices]
    correct_index = indices.index(0)  # where did the correct answer end up?

    # format prompt with question and choices
    choices_text = "\n".join(
        [f"{chr(65 + i)}. {c}" for i, c in enumerate(shuffled_choices)]
    )
    prompt = f"{row['Question']}\n\n{choices_text}"

    # the answer is the letter corresponding to the correct choice
    answer = chr(65 + correct_index)

    record = {
        # gpqa has no native id; use row index as a stable string id
        "task_id": f"GPQA/{idx}",
        "prompt": prompt,
        "answer": answer,
    }
    gpqa_records.append(record)

gpqa_records = _filter_by_length(gpqa_records, "reasoning/gpqa")
save_jsonl(gpqa_records, str(REASONING_DIR / "gpqa.jsonl"))
print(f"saved {len(gpqa_records)} records to reasoning/gpqa.jsonl")

loaded 448 problems
  reasoning/gpqa: all 448 prompts within token limit
saved 448 records to reasoning/gpqa.jsonl


## Summary

All datasets have been downloaded and formatted. Run inference with:

```bash
# baseline (single-pass) inference
infer -m qwen3-8b -d code/evalplus --baseline
infer -m qwen3-8b -d math/gsm8k --baseline
infer -m qwen3-8b -d math/math500 --baseline
infer -m qwen3-8b -d reasoning/gpqa --baseline

# two-pass overflow inference
infer -m qwen3-8b -d code/evalplus --max-think-tokens 8192 --overflow-suffix formal
```

In [21]:
# list all downloaded datasets
print("downloaded datasets:")
for d in [CODE_DIR, CRUX_DIR, MATH_DIR, REASONING_DIR]:
    for f in sorted(d.glob("*.jsonl")):
        with open(f) as fp:
            count = sum(1 for _ in fp)
        print(f"  {f}: {count} records")

downloaded datasets:
  code/bigcodebench.jsonl: 1140 records
  code/code_contests.jsonl: 165 records
  code/evalplus.jsonl: 542 records
  code/livecodebench.jsonl: 342 records
  crux/cruxeval_i.jsonl: 800 records
  crux/cruxeval_o.jsonl: 800 records
  math/gsm8k.jsonl: 1319 records
  math/math500.jsonl: 500 records
  reasoning/gpqa.jsonl: 448 records


In [22]:
"""Token length audit — estimated token counts per dataset (1 token ≈ 4 chars).

Shows p50/p95/max so we can spot datasets with prompts too long for our models.
Our context window is 32768 tokens; prompts should ideally be well under half
that to leave room for thinking tokens and an answer.
"""

print(
    f"Token budget: {MAX_PROMPT_TOKENS} max prompt tokens (half the 32768-token context window)"
)
print(
    f"{'dataset':<30}  {'count':>6}  {'p50':>6}  {'p95':>6}  {'max':>6}  {'>limit':>7}"
)
print("-" * 70)

all_dirs = [CODE_DIR, CRUX_DIR, MATH_DIR, REASONING_DIR]
for d in all_dirs:
    for f in sorted(d.glob("*.jsonl")):
        # load all prompts from this dataset file
        with open(f) as fp:
            records = [json.loads(line) for line in fp if line.strip()]

        # estimate token counts for each prompt
        tok_counts = [_estimate_tokens(r["prompt"]) for r in records]

        p50 = int(statistics.median(tok_counts))
        p95 = int(sorted(tok_counts)[int(len(tok_counts) * 0.95)])
        max_tok = max(tok_counts)
        over_limit = sum(1 for t in tok_counts if t > MAX_PROMPT_TOKENS)

        # flag datasets with any prompts over the limit
        flag = " !" if over_limit else ""
        name = str(f.parent.name) + "/" + f.stem
        print(
            f"  {name:<28}  {len(records):>6}  {p50:>6}  {p95:>6}  {max_tok:>6}  {over_limit:>6}{flag}"
        )

Token budget: 4096 max prompt tokens (half the 32768-token context window)
dataset                          count     p50     p95     max   >limit
----------------------------------------------------------------------
  code/bigcodebench               1140     151     301     866       0
  code/code_contests               165     488     894    1239       0
  code/evalplus                    542      36     170     350       0
  code/livecodebench               342     350     672     972       0
  crux/cruxeval_i                  800     239     266     274       0
  crux/cruxeval_o                  800     226     254     260       0
  math/gsm8k                      1319      55     107     212       0
  math/math500                     500      37     129     433       0
  reasoning/gpqa                   448     144     343    1406       0
